# Phase 3: Baseline Ranking System and Failure@K Definition

Objective: Establish a simple, interpretable baseline ranking system and formally define Failure@K to provide a reference point for the analysis. 

## Phase 3 Goals:
1. Formally define Failure@K
2. Implement simple baseline ranker 
3. Evaluate baseline with P@K and NDCG@K for K={1,3,5,10}
4. Initial failure analysis
5. Output for later phases

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ndcg_score
import warnings
warnings.filterwarnings('ignore')

#Display settings
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',100)
pd.set_option('display.float_format','{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi']=100

#setup paths
PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT/'data'/'raw'
DATA_PROCESSED=PROJECT_ROOT/'data'/'processed'

#Creating outputs directory
PHASE3_OUTPUT=DATA_PROCESSED/'phase3_baseline'
PHASE3_OUTPUT.mkdir(parents=True,exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Phase 3 output directory: {PHASE3_OUTPUT}")

Project root: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding
Phase 3 output directory: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase3_baseline


# Loading Data

In [3]:
#Loading processed training data from Phase1
train_file=DATA_PROCESSED/'fold1_train_processed.csv'
print(f"Loading training data from: {train_file}")
train_df=pd.read_csv(train_file)

#Loading phase 2 feature statistics
feature_stats_file=DATA_PROCESSED/'fold1_feature_statistics.csv'
print(f"Loading feature statistics from: {feature_stats_file}")
feature_stats=pd.read_csv(feature_stats_file, index_col=0)

print(f"\nTraining data loaded: {train_df.shape}")
print(f"Feature statistics loaded: {feature_stats.shape}")

#Identifying feature columns
feature_cols = [col for col in train_df.columns if col.startswith('f') and col[1:].isdigit()]
feature_cols=sorted(feature_cols,key=lambda x: int(x[1:]))
num_features=len(feature_cols)

print(f"\nDataset overview:")
print(f"Features: {num_features}")
print(f"Documents: {len(train_df):,}")
print(f"Queries: {train_df['qid'].nunique():,}")
print(f"Labels: {sorted(train_df['label'].unique())}")

Loading training data from: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\fold1_train_processed.csv
Loading feature statistics from: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\fold1_feature_statistics.csv

Training data loaded: (42158, 49)
Feature statistics loaded: (46, 15)

Dataset overview:
Features: 46
Documents: 42,158
Queries: 1,017
Labels: [np.int64(0), np.int64(1), np.int64(2)]


## Loading Test Data

In [4]:
#We need to load and parse the test data
#Using the same parser from Phase 1

def parse_letor_file(filepath):
    data=[]
    with open(filepath, 'r') as f:
        for line_num, line in enumerate(f,1):
            line=line.strip()
            if not line:
                continue

            try:
                #Splitting comment (docid) from features
                parts=line.split("#")
                feature_part=parts[0].strip()

                #Extracting docid if present
                docid=None
                if len(parts)>1:
                    comment=parts[1].strip()
                    if 'docid' in comment:
                        docid=comment.split('=')[1].strip()

                #Parsing feature part
                tokens=feature_part.split()
                
                #Extracting label
                label=int(tokens[0])

                #Extracting qid
                qid_str=tokens[1]
                qid=int(qid_str.split(':')[1])

                #Extracting features
                features={}
                for token in tokens[2:]:
                    fid, value=token.split(':')
                    features[int(fid)]=float(value)

                data.append({
                    'label':label,
                    'qid':qid,
                    'features':features,
                    'docid':docid
                })

            except Exception as e:
                print(f"Error parsing line {line_num}: {e}")
                continue

    return data

#Loading test data
FOLD1_DIR=DATA_RAW/'MQ2007'/'Fold1'
TEST_FILE=FOLD1_DIR/'test.txt'

if not TEST_FILE.exists():
    print("Test file not found")

print(f"\nLoading test data from: {TEST_FILE}")
print(f"File exists: {TEST_FILE.exists()}")

if TEST_FILE.exists():
    test_data=parse_letor_file(str(TEST_FILE))

    #Converting to dataframe
    records=[]
    for item in test_data:
        record={
            'label':item['label'],
            'qid':item['qid'],
            'docid':item['docid']
        }
        for fid, value in item['features'].items():
            record[f'f{fid}']=value
        records.append(record)

    test_df=pd.DataFrame(records)
    print(f"Test data loaded: {test_df.shape}")
    print(f"Queries: {test_df['qid'].nunique():,}")
    print(f"Documents: {len(test_df):,}")

else:
    print(f"Test file not found. Please verify the path.")
    print(f"Expected location: {FOLD1_DIR}")


Loading test data from: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\raw\MQ2007\Fold1\test.txt
File exists: True
Test data loaded: (13652, 49)
Queries: 336
Documents: 13,652


## Phase 2 findings

In [ ]:
print("="*60)
print("Phase 2 Findings Review")
print("="*60)

#Identifyiing zero variance features to exclude
zero_var_features=feature_stats[feature_stats['variance']==0].index.tolist()
print(f"\n1. Zero-variance features (to be excluded)")
print(f"Count:{len(zero_var_features)}")
if zero_var_features:
    print(f"Features: {zero_var_features}")
    print(f"Decision: These will be excluded from baseline")
else:
    print(f"No zero-variance features detected")

#Identifying most discriminative features from Phase 2
#We'll look at the features with high variance and good label separation
non_zero_features=[f for f in feature_cols if f not in zero_var_features]

print(f"\n2. Feature Screening for Baseline")
print(f"Total features: {num_features}")
print(f"Non-zero variance features: {len(non_zero_features)}")

#Computing feature discriminative power (simple mean difference between label 0 and 2)
if len(non_zero_features)>0:
    discriminative_power={}
    for feat in non_zero_features:
        label_0_mean=train_df[train_df['label']==0][feat].mean()
        label_2_mean=train_df[train_df['label']==2][feat].mean()
        discriminative_power[feat]=abs(label_2_mean-label_0_mean)

    #Sorting by discriminative power
    sorted_features=sorted(discriminative_power.items(), key=lambda x: x[1], reverse=True)
    print(f"\nTop 10 most discriminative features (by |mean(label=2)-mean(label=0)|):")
    for feat, power in sorted_features[:10]:
        print(f"{feat}:{power:.4f}")

    most_discriminative_feature=sorted_features[0][0]
    print(f"\nMost discriminative feature: {most_discriminative_feature}")

print(f"\n3. Variance Characteristics")
print(f"Mean feature variance: {feature_stats.loc[non_zero_features, 'variance'].mean():.4f}")
print(f"Mean feature sparsity: {feature_stats.loc[non_zero_features, 'pct_zeros'].mean():.2f}%")
print(f"\nPhase 2 conclusion: Strong within-query variance observed")
print(f"Phase 3 decision: Start with raw features (test normalization later)")

Phase 2 Findings Review

1. Zero-variance features (to be excluded)
Count:5
Features: ['f6', 'f7', 'f8', 'f9', 'f10']
Decision: These will be excluded from baseline

2. Feature Selection for Baseline
Total features: 46
Non-zero variance features: 41

Top 10 most discriminative features (by |mean(label=2)-mean(label=0)|):
f23:0.1803
f39:0.1782
f21:0.1570
f37:0.1526
f40:0.1350
f38:0.1255
f24:0.1252
f22:0.1240
f25:0.1024
f31:0.0971

Most discriminative feature: f23

3. Variance Characteristics
Mean feature variance: 0.0760
Mean feature sparsity: 36.83%

Phase 2 conclusion: Strong within-query variance observed
Phase 3 decision: Start with raw features (test normalization later)


We identified in phase 2 that there are 5 zero-variance features (f6-f10), which are now excluded from modeling because they provide no discriminative signal. 

The discriminative features that we listed are candidates for signal, not some guaranteed predictors.